In [1]:
import os

os.chdir('..')

## Loading the Dataset:

In this section the pointcloud is loaded. The SIREN paper suggests normalizing the point coordinates as periodic activations implicitly expect a bounded input. 

In [2]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import src.model.SIREN as si
from src.model.training import train
import src.loss.SDF_loss as loss
from src.mesh_extraction.marching_cubes_test import write_obj
import src.model.MLP as simple
import src.data.dataset as data

mesh = data.MeshDataset('data/pointclouds/Armadillo/Armadillo.ply')
print(mesh.vertices)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[[ 0.07790768 -0.12729204  0.36060192]
 [-0.7047902   0.60381086 -0.75930651]
 [ 0.04956958 -0.06494814  0.3892928 ]
 ...
 [-0.00709846 -0.36533849  0.56706102]
 [-0.65767778 -0.97143663  0.01645928]
 [-0.76626174  0.54733239 -0.66349749]]


## Defining the Model

In this cell we will define the SIREN model. This particular INR uses sine activations for nonlinearity and is supposed to capture more information given the underlying data when compared to a model that uses ReLU activations. This way, a good INR accuracy can be achieved with fewer neurons.

In [3]:
size_per_layer = 256
model = si.SIRENSDF(num_hidden_layers=4)
model_loss = loss.Loss(lambda_sdf=1, lambda_surface=1,  lambda_eikonal=1, lambda_normal=0, lambda_twd=1e-4, k=int(size_per_layer/5), model=model) # optional normal loss if normals contained in the pointcloud
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


## Model training



In [4]:
train(epochs=2000, data=mesh, no_surface=10000, no_off_surface=15000, model=model, loss=model_loss, optimizer=optimizer, prune=False, scene=mesh.scene)

Step 0 | Loss 2.4279747009277344
Step 10 | Loss 0.46536844968795776
Step 20 | Loss 0.3552849590778351
Step 30 | Loss 0.30585119128227234
Step 40 | Loss 0.27728593349456787
Step 50 | Loss 0.25351014733314514
Step 60 | Loss 0.23954975605010986
Step 70 | Loss 0.23010621964931488
Step 80 | Loss 0.22244766354560852
Step 90 | Loss 0.21866734325885773
Step 100 | Loss 0.21113252639770508
Step 110 | Loss 0.20595571398735046
Step 120 | Loss 0.20223775506019592
Step 130 | Loss 0.19905579090118408
Step 140 | Loss 0.1952248364686966
Step 150 | Loss 0.192448690533638
Step 160 | Loss 0.18917742371559143
Step 170 | Loss 0.18750907480716705
Step 180 | Loss 0.18214450776576996
Step 190 | Loss 0.18021038174629211
Step 200 | Loss 0.17954707145690918
Step 210 | Loss 0.17633825540542603
Step 220 | Loss 0.17433568835258484
Step 230 | Loss 0.17369917035102844
Step 240 | Loss 0.17240384221076965
Step 250 | Loss 0.1707322597503662
Step 260 | Loss 0.16892707347869873
Step 270 | Loss 0.1667921245098114
Step 280 |

#### Model size after pruning

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Sample a few surface points and check their SDF values
test_points = mesh.vertices[:10]  # First 10 points
test_tensor = torch.from_numpy(test_points).float().to(device)
with torch.no_grad():
    sdf_values = model(test_tensor)
print("SDF values for surface points:")
print(sdf_values)
print("Mean absolute SDF:", torch.abs(sdf_values).mean().item())

SDF values for surface points:
tensor([[-0.0138],
        [-0.0160],
        [-0.0171],
        [-0.0188],
        [-0.0153],
        [-0.0148],
        [-0.0158],
        [-0.0168],
        [-0.0138],
        [-0.0193]], device='cuda:0')
Mean absolute SDF: 0.01614280417561531


In [6]:
import src.mesh_extraction.marching_cubes_gpu as marching_cubes
marching_cubes.write_obj("armadillo_mesh_128.obj", model=model, resolution=128, level=0.0)


In [7]:
# import src.mesh_extraction.marching_cubes_test as marching_cubes_test
# marching_cubes_test.write_obj("armadillo_mesh_ground_truth_128.obj", scene=mesh.scene, resolution=128, level=0.0)


In [8]:
# Make batched point
test_point = torch.from_numpy(np.array([[-1, -1, -1]])).float().to(device)

# Compute SIREN model prediction
sdf_pred = model(test_point)
print("SIREN prediction:", sdf_pred)

# Compute Open3D signed distance
distance = mesh.scene.compute_signed_distance(
    o3d.core.Tensor(test_point.cpu().numpy(), dtype=o3d.core.Dtype.Float32)
)
print("Open3D SDF:", distance.numpy())

SIREN prediction: tensor([[0.0723]], device='cuda:0', grad_fn=<AddmmBackward0>)
Open3D SDF: [0.96991843]
